# Extra runs: RQ2 stated answers on ScienceWorld and the extra items; horizon ablation on 3 splits

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **on**, Persistence = **Files only**.
**Secret:** Add-ons -> Secrets -> `HF_TOKEN` (your Hugging Face token; needed for Llama).
**Input:** + Add Input -> Datasets -> your uploaded dataset `irrev-extra-runs` (contains `irrev_extra_runs_package.zip`).

Run **one model per session**: set `MODEL`/`TAG` in cell 3, run cells 1-7 top to bottom, download the zip from cell 7.
Then start a new session (or Run All again) with the other model.
Every step resumes where it stopped if the session dies: just run cells 1-3 again, then the step that was cut off.

In [ ]:
# 1. unpack the code + data
import subprocess, os, glob
z = glob.glob('/kaggle/input/**/irrev_extra_runs_package.zip', recursive=True)
d = glob.glob('/kaggle/input/**/irrev_pkg', recursive=True)
if not os.path.exists('/kaggle/working/irrev_pkg'):
    if d: subprocess.run(f'cp -r "{d[0]}" /kaggle/working/irrev_pkg', shell=True)
    else: subprocess.run(f'cd /kaggle/working && unzip -q "{z[0]}"', shell=True)
%cd /kaggle/working/irrev_pkg
!pip install -q "transformers==5.22.1" "accelerate==1.15.0" "scikit-learn==1.6.1"
!python -c "import torch,transformers,sklearn;print('torch',torch.__version__,'| transformers',transformers.__version__,'| sklearn',sklearn.__version__,'|',torch.cuda.device_count(),'GPUs')"


In [ ]:
# 2. Hugging Face login (Llama is gated; Qwen also works without it)
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
!hf auth login --token $HF_TOKEN 2>/dev/null || huggingface-cli login --token $HF_TOKEN


In [ ]:
# 3. choose the model for THIS session
MODEL, TAG, NEWTOK = 'Qwen/Qwen2.5-7B-Instruct', 'qwen7b', 96
# MODEL, TAG, NEWTOK = 'meta-llama/Llama-3.1-8B-Instruct', 'llama8b', 256     # second session
R = f'results/{TAG}'
print(MODEL, R)


In [ ]:
# 4. RQ2: stated answers on ScienceWorld (626 items) and the extra binding items (716)  ~20-35 min each
!python -m irrev.stated --model $MODEL --data data/probe_scienceworld.jsonl --out $R/stated_sw.jsonl --batch 8 --max-new-tokens $NEWTOK
!python -m irrev.stated --model $MODEL --data data/probe_sandbox_extra.jsonl --out $R/stated_extra.jsonl --batch 8 --max-new-tokens $NEWTOK
!cd /kaggle/working/irrev_pkg && zip -qr /kaggle/working/extra_runs_$TAG.zip $R/stated_sw* $R/stated_extra*
print('saved /kaggle/working/extra_runs_'+TAG+'.zip (download it now as a safety copy)')


In [ ]:
# 5. hidden states for all probe items (needed for the probe refit)  ~20-30 min
!python -m irrev.extract --model $MODEL --data data/probe_sandbox.jsonl       --out $R/hidden_sandbox
!python -m irrev.extract --model $MODEL --data data/probe_sandbox_extra.jsonl --out $R/hidden_extra
!python -m irrev.extract --model $MODEL --data data/probe_scienceworld.jsonl  --out $R/hidden_sw


In [ ]:
# 6. horizon ablation on all 3 splits (CPU, ~1-1.5 h). Every finished sweep in probes/partial is reused;
#    only the new *_h1_3seeds / *_h2_3seeds sweeps are fitted.
!python -m irrev.probes --out $R/probes --horizon-all-seeds \
   --set sandbox:$R/hidden_sandbox:data/probe_sandbox.jsonl \
   --set sandbox:$R/hidden_extra:data/probe_sandbox_extra.jsonl \
   --set scienceworld:$R/hidden_sw:data/probe_scienceworld.jsonl


In [ ]:
# 7. package everything for download (no hidden states: they are ~1.5 GB)
!cd /kaggle/working/irrev_pkg && rm -f /kaggle/working/extra_runs_$TAG.zip && zip -qr /kaggle/working/extra_runs_$TAG.zip $R/stated_sw* $R/stated_extra* $R/probes/probes.json $R/probes/partial/*_3seeds.json
!unzip -l /kaggle/working/extra_runs_$TAG.zip
print('Download /kaggle/working/extra_runs_'+TAG+'.zip from the Output panel (right side, /kaggle/working).')
